# Use SpecStream web search with Mistral

Give Mistral models live web search and page extraction through SpecStream using function calling.

*Author: Tarek Etman (SpecStream), GitHub: [Specstreamai](https://github.com/Specstreamai)*

<a href="https://colab.research.google.com/github/mistralai/cookbook/blob/main/third_party/SpecStream/specstream_web_search_with_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Language models only reflect information present in their training data unless you connect them to external tools. [SpecStream](https://specstream.ai) provides a live web Search API (`GET /v1/web-search`) and an Extract API (`POST /v1/extract`) that return ranked results, page passages, and source URLs with read timestamps.

In this cookbook, you wrap SpecStream's Search and Extract endpoints as Python functions, register them as tools with the Mistral Python SDK, and run a multi-turn tool-calling loop with `mistral-small-latest` to answer questions grounded in current web sources.

## Prerequisites

To complete this cookbook, you will need:

- Python 3.10 or later (or Google Colab)
- A Mistral account and API key
- A SpecStream account and an API key with the `finds:spend` scope

## Environment setup

This cookbook is pinned to `mistralai` 3.7.0 and `requests` 2.34.2 so it runs consistently in Google Colab and local Jupyter environments.

### Install

Install the Mistral Python SDK and `requests`:

In [ ]:
%pip install "mistralai==3.7.0" "requests==2.34.2" --quiet

### Required environment variables

To complete this cookbook, you'll need a Mistral API key. In [Studio](https://console.mistral.ai), navigate to the [API keys section](https://console.mistral.ai/home?profile_dialog=api-keys) and create a new API key.

You'll also need a SpecStream API key (`SPECSTREAM_API_KEY`). Create one under [Workspace > Keys](https://specstream.ai/workspace/keys) in SpecStream with the `finds:spend` scope and a spending limit above `0`.

Provide both keys in one of two ways:

1. Set them as environment variables before you start the notebook:

   ```
   MISTRAL_API_KEY=your-mistral-api-key
   SPECSTREAM_API_KEY=your-specstream-api-key
   ```

2. Leave them unset and enter them when the next cell prompts you.

In [ ]:
import getpass
import os

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")

if not os.environ.get("SPECSTREAM_API_KEY"):
    os.environ["SPECSTREAM_API_KEY"] = getpass.getpass("SpecStream API key: ")

## 1. Call SpecStream search and extract directly

Before wiring tools into Mistral, define small helper functions for SpecStream's Search API (`GET /v1/web-search`) and Extract API (`POST /v1/extract`).

In `specstream_web_search()`:

- `speed="fast"` runs the fast search tier ($1 per 1,000 searches of up to 10 results; omit `speed` for the default tier at $5 per 1,000).
- `passages="true"` returns page excerpts (`excerpt` on each source and top-level `passages`) so the model has concrete text without fetching every URL separately.

Define the shared request helper, `specstream_web_search()`, and `specstream_extract()`:

In [ ]:
from typing import Any
import requests

SPECSTREAM_BASE_URL = "https://api.specstream.ai"


def _specstream_headers() -> dict[str, str]:
    """Return authenticated headers for SpecStream API calls."""
    api_key = os.environ.get("SPECSTREAM_API_KEY", "").strip()
    if not api_key:
        raise ValueError("Set SPECSTREAM_API_KEY before calling SpecStream.")
    return {"Authorization": f"Bearer {api_key}"}


def specstream_web_search(
    query: str,
    num: int = 5,
    speed: str = "fast",
    include_passages: bool = True,
) -> dict[str, Any]:
    """Search the live web with SpecStream and return sources and optional passages."""
    params: dict[str, Any] = {
        "q": query,
        "num": max(1, min(num, 20)),
    }
    if speed:
        params["speed"] = speed
    if include_passages:
        params["passages"] = "true"

    response = requests.get(
        f"{SPECSTREAM_BASE_URL}/v1/web-search",
        headers=_specstream_headers(),
        params=params,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json().get("data", {})

    sources = [
        {
            "title": item.get("title"),
            "url": item.get("url"),
            "site": item.get("site"),
            "published": item.get("published"),
            "snippet": item.get("snippet"),
            "excerpt": item.get("excerpt"),
        }
        for item in data.get("sources", [])
    ]
    return {
        "query": query,
        "answer": (data.get("answer") or {}).get("value") if isinstance(data.get("answer"), dict) else data.get("answer"),
        "sources": sources,
        "took_ms": data.get("took_ms"),
        "price_usd": data.get("price_usd"),
    }


def specstream_extract(urls: list[str]) -> dict[str, Any]:
    """Extract main page text from one to five URLs using SpecStream."""
    if not urls:
        raise ValueError("Provide at least one URL to extract.")

    response = requests.post(
        f"{SPECSTREAM_BASE_URL}/v1/extract",
        headers={**_specstream_headers(), "Content-Type": "application/json"},
        json={"urls": urls[:5]},
        timeout=45,
    )
    response.raise_for_status()
    data = response.json().get("data", {})

    results = [
        {
            "url": item.get("url"),
            "status": item.get("status"),
            "title": item.get("title"),
            "published": item.get("published"),
            "read_from": item.get("read_from"),
            "text": (item.get("text") or "")[:4000],
        }
        for item in data.get("results", [])
    ]
    return {
        "results": results,
        "took_ms": data.get("took_ms"),
        "price_usd": data.get("price_usd"),
    }

Run a quick search to confirm your `SPECSTREAM_API_KEY` works and inspect the returned fields:

In [ ]:
import json

sample = specstream_web_search("Mistral AI releases documentation", num=3)
print(f"Returned {len(sample['sources'])} sources in {sample['took_ms']} ms (price: ${sample['price_usd']})")
for idx, src in enumerate(sample["sources"], start=1):
    print(f"{idx}. {src['title']} ({src['url']})")

The cell prints the count, latency, call cost, and ranked source titles with URLs. Exact results vary as live pages change:

```text
Returned 3 sources in 640 ms (price: $0.001)
1. Mistral AI Documentation (https://docs.mistral.ai/)
2. Models Overview | Mistral AI (https://docs.mistral.ai/getting-started/models/models_overview/)
3. News | Mistral AI (https://mistral.ai/news)
```

## 2. Register SpecStream as Mistral tools

Declare JSON schemas for `search_web` and `extract_pages`, along with a handler map that dispatches tool calls by name. Keeping a single dispatch table lets the tool loop execute any registered tool uniformly.

In [ ]:
from typing import Callable

TOOLS: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": (
                "Search the live web for up-to-date facts, documentation, news, or URLs. "
                "Supports search operators such as site:, filetype:, after:, and quoted phrases."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query in plain words or with operators like site:docs.mistral.ai.",
                    },
                    "num": {
                        "type": "integer",
                        "description": "Number of sources to return (1 to 10).",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "extract_pages",
            "description": (
                "Read the main text of up to 5 web pages when search excerpts are not enough "
                "to answer a question completely."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "urls": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of public http or https URLs to read.",
                    }
                },
                "required": ["urls"],
            },
        },
    },
]

TOOL_HANDLERS: dict[str, Callable[..., dict[str, Any]]] = {
    "search_web": lambda query, num=5: specstream_web_search(query=query, num=num),
    "extract_pages": lambda urls: specstream_extract(urls=urls),
}

## 3. Run the tool-calling loop with Mistral

Initialize the Mistral client and write a loop that passes `TOOLS` to `client.chat.complete()`. When `mistral-small-latest` requests a tool call, the loop executes the matching SpecStream function, appends the result as a `tool` message, and asks the model for the final grounded response.

In [ ]:
from mistralai.client import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

SYSTEM_PROMPT = (
    "You are a research assistant with live web search and page extraction tools. "
    "Use search_web to verify current facts before answering, and cite source URLs inline."
)


def run_tool_call(name: str, raw_arguments: str) -> str:
    """Execute a registered SpecStream tool and serialize its output as JSON."""
    handler = TOOL_HANDLERS.get(name)
    if handler is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    args = json.loads(raw_arguments)
    result = handler(**args)
    return json.dumps(result)


def answer_with_web_search(
    question: str,
    model: str = "mistral-small-latest",
    max_turns: int = 5,
) -> str:
    """Answer a question using Mistral function calling backed by SpecStream."""
    messages: list[Any] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for _ in range(max_turns):
        response = client.chat.complete(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        assistant_msg = response.choices[0].message
        messages.append(assistant_msg)

        if not assistant_msg.tool_calls:
            return assistant_msg.content or ""

        for tool_call in assistant_msg.tool_calls:
            name = tool_call.function.name
            print(f"[tool] {name}({tool_call.function.arguments})")
            output = run_tool_call(name, tool_call.function.arguments)
            messages.append(
                {
                    "role": "tool",
                    "name": name,
                    "content": output,
                    "tool_call_id": tool_call.id,
                }
            )

    raise RuntimeError(f"Model did not finish within {max_turns} tool-calling turns.")

Ask a question that benefits from checking live documentation (using SpecStream's `site:` operator or natural-language search):

In [ ]:
question = (
    "What endpoints and Python SDK packages does Mistral document for chat completions "
    "and function calling on docs.mistral.ai? Cite the exact documentation URLs."
)
answer = answer_with_web_search(question)
print("\n" + answer)

The notebook logs each tool call issued by the model, then prints the final cited answer. Exact wording and URLs depend on the live search results:

```text
[tool] search_web({"query": "site:docs.mistral.ai chat completions function calling Python SDK", "num": 5})

Mistral documents the `POST /v1/chat/completions` endpoint for chat completions and tool calling, accessed in Python through the `mistralai` SDK (`from mistralai.client import Mistral`).

Sources:
- https://docs.mistral.ai/capabilities/completion/
- https://docs.mistral.ai/capabilities/function_calling/
```

SpecStream also exposes drop-in endpoints (such as `https://api.specstream.ai/tavily` and `https://api.specstream.ai/exa`) and a remote MCP server at `https://api.specstream.ai/mcp` if your workflow already uses those interfaces.

## Summary

In this cookbook, you connected Mistral function calling to SpecStream's Search and Extract APIs so `mistral-small-latest` can retrieve live web passages and full page text with source URLs.

**What you built:**

- Reusable Python wrappers for SpecStream's `GET /v1/web-search` (fast tier with passages) and `POST /v1/extract` endpoints
- Tool definitions (`search_web` and `extract_pages`) registered with `client.chat.complete()`
- A multi-turn tool-calling loop that returns grounded answers with inline URL citations

**Mistral features used:**

- Chat Completions API
- Function calling (`tools` and `tool_choice`)
- Mistral Python SDK (`mistralai==3.7.0`)

**Other services:**

- [SpecStream](https://specstream.ai) Search API and Extract API ([API reference](https://specstream.ai/agents.md))

Read the [Mistral function calling guide](https://docs.mistral.ai/capabilities/function_calling/) to explore parallel tool calls and structured outputs.